# 01 · Modellantworten sammeln: GPT und Claude

**Ziel:** Für jedes Satzpaar fünf Antworten pro Modell speichern (fünf Formulierungen derselben Aufgabe).

Jede Formulierung ist ein **"Kopf"** eines kleinen Ensembles. Daraus entsteht später die Zerlegung:

| Grösse | Bedeutung | Spalte |
|---|---|---|
| Gesamtunsicherheit | Entropie der gemittelten Verteilung | `mean_entropy` |
| = erwartete Entropie | Wie unentschieden ist jeder einzelne Kopf? (Modellsicht auf **"debatable"**) | `expected_entropy` |
| + Uneinigkeit zwischen Köpfen | Widersprechen sich die Köpfe? (**"uncertain"**, epistemisch) | `epistemic_mi` |
| Credal-Intervall | [min, max] pro Label über die Köpfe, Breite = epistemisches Signal | `credal_width` |

Das Modell erhält nur Ausgangssatz und Hypothese. Menschliche Stimmen und Kategorien werden nie mitgeschickt.
Prompt-Variation ist ein messbarer Stellvertreter für epistemische Unsicherheit, kein Beweis dafür.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config/project.json").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display, Image
from uncertainty_project.common import config
CFG = config(ROOT)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
print("Projekt:", ROOT.name, "| Modelle:", list(CFG["models"]))

Projekt: uncertainty_project | Modelle: ['gpt', 'claude']


In [2]:
from dotenv import dotenv_values
print([f.name for f in ROOT.iterdir() if "env" in f.name])
vals = dotenv_values(ROOT / ".env")
print({k: len(v or "") for k, v in vals.items()})

['.env', '.env.example']
{'OPENAI_API_KEY': 164, 'ANTHROPIC_API_KEY': 106}


In [ ]:
from dotenv import dotenv_values
k = dotenv_values(ROOT / ".env")["OPENAI_API_KEY"]
print(k[:8], "...", k[-4:], "| Länge:", len(k))

from openai import OpenAI
try:
    ids = [m.id for m in OpenAI(api_key=k).models.list()]
    print("Key funktioniert. gpt-6.1-sol verfügbar:", "gpt-6.1-sol" in ids)
except Exception as e:
    print("Fehler:", type(e).__name__)

## Stufe wählen

1. `"smoke"`: 5 Satzpaare × 2 Modelle × 5 Prompts = **50 Calls**. Prüft Schlüssel, Format und Speicherung.
2. `"pilot"`: 50 Satzpaare × 2 Modelle × 5 Prompts = **500 Calls** (die 5 aus dem Smoke-Test werden wiederverwendet).
3. `"train_dev"`: alle Trainings- und Entwicklungsbeispiele, ca. **6'360 Calls pro Modell**.

`ALLOW_PAID_API = False` zeigt nur den Plan. Schlüssel gehören in `.env` (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`).
Calls laufen parallel (`api.parallel_requests` in der Config). Abgebrochene Läufe einfach neu starten: Gespeichertes wird wiederverwendet.

In [15]:
from uncertainty_project.data import load_split, pilot_rows, smoke_rows
from uncertainty_project.api import request_plan, collect_predictions, usage_report, run_directory

STAGE = "train_dev"            # "smoke" -> "pilot" -> "train_dev"
MODELS = ["gpt", "claude"]  # einzeln testen geht auch, z.B. ["claude"]
ALLOW_PAID_API = True
MAX_NEW_REQUESTS = 6400       # pro Modell; für train_dev auf ca. 6400 setzen

if STAGE == "smoke":
    batch = smoke_rows(ROOT)
elif STAGE == "pilot":
    batch = pilot_rows(ROOT)
elif STAGE == "train_dev":
    batch = pd.concat([load_split(ROOT, "train"), load_split(ROOT, "dev")], ignore_index=True)
else:
    raise ValueError("Wähle smoke, pilot oder train_dev. Testfälle werden erst in Notebook 03 abgefragt.")
display(pd.DataFrame([request_plan(ROOT, batch, k) for k in MODELS]))

,model_key,model,examples,variants,logical_requests,cached,new_requests,max_output_tokens_per_request,note
0,gpt,gpt-6.1-sol,1272,5,6360,250,6110,4096,Calls use paid API access. Estimate full-study...
1,claude,claude-sonnet-5-5,1272,5,6360,250,6110,4096,Calls use paid API access. Estimate full-study...


In [17]:
results = []
for key in MODELS:
    results.append(collect_predictions(ROOT, batch, key, allow_api=ALLOW_PAID_API, max_new_requests=MAX_NEW_REQUESTS))
display(pd.DataFrame(results))

,model_key,model,examples,variants,logical_requests,cached,new_requests,max_output_tokens_per_request,note,mode
0,gpt,gpt-6.1-sol,1272,5,6360,6360,0,4096,Calls use paid API access. Estimate full-study...,cache_only
1,claude,claude-sonnet-5-5,1272,5,6360,6360,0,4096,Calls use paid API access. Estimate full-study...,cache_only


## Verbrauch und Kostenhochrechnung

Hochrechnung auf alle 1'599 Beispiele (Train + Dev + Test) aus dem gemessenen Verbrauch.
Fehlgeschlagene Antworten werden **nie** ergänzt oder erfunden. Wiederholte Versuche werden im Cache protokolliert.

In [18]:
total_examples = sum(len(load_split(ROOT, s)) for s in ("train", "dev", "test"))
rows = []
for key in MODELS:
    u = usage_report(ROOT, batch, key)
    if u["successful_cached_requests"] == len(batch) * 5:
        u["hochrechnung_alle_beispiele_usd"] = round(u["estimated_usd_per_example"] * total_examples, 2)
    rows.append(u)
display(pd.DataFrame(rows).drop(columns="note"))

,model_key,successful_cached_requests,requests_needing_retry,input_tokens,output_tokens,estimated_usd,estimated_usd_per_example,hochrechnung_alle_beispiele_usd
0,gpt,6360,0,2495277,842364,13.414194,0.010546,16.86
1,claude,6360,4,5096030,977455,19.966610,0.015697,25.10


## Ein Beispiel im Detail: fünf Köpfe pro Modell

In [19]:
from uncertainty_project.api import read_cached
from uncertainty_project.features import summarize_predictions

first = batch.iloc[0].to_dict()
print("Ausgangssatz:", first["premise"])
print("Hypothese:   ", first["hypothesis"])
for key in MODELS:
    heads = [read_cached(ROOT, first, v, key) for v in range(5)]
    if all(h is not None for h in heads):
        print(f"\n=== {key} ({heads[0].get('returned_model', '?')})")
        display(pd.DataFrame([h["parsed"] for h in heads]))
        s = summarize_predictions([h["parsed"] for h in heads])
        display(pd.Series({k: round(s[k], 3) for k in ("mean_entropy", "expected_entropy", "epistemic_mi",
                           "credal_width", "credal_determinate", "answer_disagreement", "clarifiable_cue")}))
    else:
        print(f"{key}: noch keine vollständigen Antworten.")

Ausgangssatz: No one would ever think of sentiment in connection with you.
Hypothese:    No one would think of that because it's not who you are.

=== gpt (gpt-6.1-sol)


,entailment,neutral,contradiction,lexical_ambiguity,unclear_reference,probabilistic_inference,brief_reason
0,0.36,0.63,0.01,0.20,0.35,0.75,"If “that” refers to sentiment, both statements..."
1,0.35,0.64,0.01,0.20,0.60,0.70,"If ""that"" refers to sentiment, both statements..."
2,0.58,0.41,0.01,0.22,0.35,0.72,"If 'that' refers to sentiment, the hypothesis ..."
3,0.38,0.61,0.01,0.18,0.32,0.72,"If ""that"" refers to sentiment, the premise sup..."
4,0.55,0.44,0.01,0.18,0.56,0.72,"If “that” refers to sentiment, the hypothesis ..."


mean_entropy           0.671
expected_entropy       0.652
epistemic_mi           0.019
credal_width           0.153
credal_determinate     0.000
answer_disagreement    0.400
clarifiable_cue        0.436
dtype: float64


=== claude (claude-sonnet-5-5)


,entailment,neutral,contradiction,lexical_ambiguity,unclear_reference,probabilistic_inference,brief_reason
0,0.15,0.70,0.15,0.10,0.35,0.6,The premise says no one associates sentiment w...
1,0.12,0.80,0.08,0.10,0.20,0.6,The premise says nobody associates sentiment w...
2,0.12,0.78,0.10,0.08,0.20,0.6,The premise says nobody associates sentiment w...
3,0.20,0.68,0.12,0.10,0.25,0.6,The premise says no one associates sentiment w...
4,0.15,0.70,0.15,0.10,0.20,0.6,The premise says no one associates sentiment w...


mean_entropy           0.697
expected_entropy       0.690
epistemic_mi           0.007
credal_width           0.090
credal_determinate     1.000
answer_disagreement    0.000
clarifiable_cue        0.240
dtype: float64

## Pilot ansehen (nur Trainingsdaten)

Die interessanten Fälle für den Bericht und das nächste Gespräch mit Christina:
- **Menschen einig, Modell weicht ab** → Kandidaten für *uncertain*
- **Menschen gespalten** → Kandidaten für *debatable*
- **GPT und Claude antworten verschieden**

In [20]:
import numpy as np
from uncertainty_project.common import LABELS
from uncertainty_project.features import feature_frame, entropy, majority_correct

if STAGE in ("smoke", "pilot"):
    try:
        feats = {k: feature_frame(ROOT, batch, k).set_index("uid") for k in MODELS}
        q = np.stack(batch.set_index("uid").loc[feats[MODELS[0]].index, "human_distribution"])
        view = pd.DataFrame({"human_entropy": entropy(q).round(2),
                             "human_majority": [LABELS[i][:4] for i in q.argmax(1)]}, index=feats[MODELS[0]].index)
        for k in MODELS:
            f = feats[k]
            view[f"{k}_answer"] = [LABELS[i][:4] for i in f.base_label]
            view[f"{k}_ok"] = majority_correct(f.base_label, q).astype(int)
            view[f"{k}_epistemic_mi"] = f.epistemic_mi.round(3)
            view[f"{k}_expected_H"] = f.expected_entropy.round(2)
        view = view.join(batch.set_index("uid")[["premise", "hypothesis"]])
        print("Übereinstimmung mit der Mehrheit:", {k: float(view[f"{k}_ok"].mean()) for k in MODELS})
        # Same fixed rule as later: quantile of human entropy on the full TRAIN split, not on the pilot.
        train_h = entropy(np.stack(load_split(ROOT, "train").human_distribution))
        clear_max = float(np.quantile(train_h, CFG["decomposition"]["clear_case_train_entropy_quantile"]))
        clear = view.human_entropy <= clear_max
        print(f"Klare Fälle (Entropie <= {clear_max:.3f}, Train-Schwelle): {int(clear.sum())} von {len(view)}")
        for k in MODELS:
            print(f"  {k}: Mehrheitsabweichungen auf klaren Fällen = {int((view.loc[clear, f'{k}_ok'] == 0).sum())}")
        print("\nMenschen einig, mindestens ein Modell daneben (uncertain-Kandidaten):")
        display(view[clear & (view[[f"{k}_ok" for k in MODELS]].min(axis=1) == 0)])
        print("\nAm stärksten gespaltene Fälle (debatable-Kandidaten):")
        display(view.sort_values("human_entropy", ascending=False).head(8))
        if len(MODELS) == 2:
            print("\nGPT und Claude antworten verschieden:")
            display(view[view[f"{MODELS[0]}_answer"] != view[f"{MODELS[1]}_answer"]])
        print("\nSpearman epistemic_mi vs. menschliche Entropie (soll eher tief sein):",
              {k: round(float(view[f"{k}_epistemic_mi"].corr(view.human_entropy, method="spearman")), 3) for k in MODELS})
        print("Spearman expected_H vs. menschliche Entropie (soll eher hoch sein):",
              {k: round(float(view[f"{k}_expected_H"].corr(view.human_entropy, method="spearman")), 3) for k in MODELS})
        print("\nPilot = Entwicklungsbeobachtung mit wenigen Fällen, keine Ergebnisse für den Bericht.")
    except RuntimeError as exc:
        print(exc)

In [21]:
   for k in MODELS:
       f = feats[k]
       print(k, "| epistemic_mi Median/90%/Max:",
             round(f.epistemic_mi.median(), 3), round(f.epistemic_mi.quantile(.9), 3), round(f.epistemic_mi.max(), 3),
             "| Anteil mit Label-Wechsel zwischen Prompts:", round(float((f.answer_disagreement > 0).mean()), 2),
             "| credal_width Median:", round(f.credal_width.median(), 3))

gpt | epistemic_mi Median/90%/Max: 0.003 0.046 0.299 | Anteil mit Label-Wechsel zwischen Prompts: 0.16 | credal_width Median: 0.053
claude | epistemic_mi Median/90%/Max: 0.008 0.041 0.144 | Anteil mit Label-Wechsel zwischen Prompts: 0.3 | credal_width Median: 0.103


**Nächster Schritt:** Nach sauberem Smoke-Test → `"pilot"`. Nach geprüftem Pilot → `"train_dev"` (läuft am besten über Nacht).
Danach Notebook 02. Der Test bleibt bis zur festgelegten Methode geschlossen.